# 23CSE365 Internet of Things – Course Project
## Explainable Kinematic Digital Biomarkers and Functional Resilience Modeling for Upper-Limb Performance Assessment in Children with Cerebral Palsy Using Markerless Motion Capture

**Institution:** Amrita Vishwa Vidyapeetham, Amrita School of Computing, Bangalore  
**Course:** 23CSE365 Internet of Things  
**Dataset:** KU Leuven Markerless Motion Capture Dataset (Vanmechelen et al., *Journal of Biomechanics*, 2024; DOI: 10.48804/BZM9SJ)  
**Standard:** Top-Tier Conference Methodology and Reproducibility Protocol

### 1. Problem Statement & Prediction Target
* **Task:** Binary Classification of Upper-Limb Motor Function.
* **Target Variable:** `target` (derived from clinical diagnostic cohort assignment).
  * **Class 1 (Positive):** Dyskinetic Cerebral Palsy (`DCP`), $N = 30$ ($58.8\%$ of cohort).
  * **Class 0 (Negative/Reference):** Typically Developing peers (`TD`), $N = 21$ ($41.2\%$ of cohort).
* **Total Participants:** $N = 51$ distinct subjects (ages 5–24 years), performing 8 repetitions of reaching-sideways.
* **Prediction Unit:** 1 row per subject (51 rows total; strictly zero cross-repetition data leakage).

In [ ]:
import os, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Load canonical experiment dataset
data_path = os.path.join("..", "data", "processed", "experiment_dataset.csv")
df = pd.read_csv(data_path)
print(f"Experiment Dataset: {df.shape[0]} subjects, {df.shape[1] - 4} kinematic predictor features")
print("Class distribution:")
print(df["group"].value_counts())
print(df[["subject", "side", "group", "target"]].head())

### 2. Full Benchmark: 16 Models with Varied Hyperparameters
All 16 classifiers are evaluated under 5-Fold Stratified Group Cross-Validation (`StratifiedGroupKFold`) with 4-fold inner CV for hyperparameter tuning. Preprocessing (median imputation, standardization, and mutual-information feature selection) is fit **strictly on training subjects** within each fold to guarantee zero data leakage.

**Crucial Methodology Rule:** Candidate models are ranked and selected strictly on **inner-validation accuracy** (`inner_accuracy_mean`) on training folds. The outer test sets are held out and never used for model selection.

In [ ]:
benchmark_path = os.path.join("..", "results", "model_benchmark", "model_benchmark_summary.csv")
bench_df = pd.read_csv(benchmark_path)
display_cols = [
    "model", "inner_accuracy_mean", "test_accuracy_mean", "test_accuracy_std",
    "test_balanced_accuracy_mean", "test_f1_mean", "test_roc_auc_mean"
]
print("=== 16-MODEL BENCHMARK TABLE (Ranked by Inner-Validation Accuracy) ===")
display_bench = bench_df[display_cols].copy()
display_bench.columns = ["Model", "Inner Acc (Train)", "Test Acc", "Test Acc SD", "Test Balanced Acc", "Test F1", "Test ROC-AUC"]
display_bench

### 3. Leak-Free Model Selection
The top 2–3 models with the highest mean inner-validation accuracy across training folds are selected for the dedicated validation stage.

In [ ]:
selection_path = os.path.join("..", "results", "model_benchmark", "selected_top_models.json")
with open(selection_path, "r", encoding="utf-8") as f:
    sel_meta = json.load(f)

print(f"Selection Criterion: {sel_meta['selection_criterion']}")
print(f"Selected Top Models: {sel_meta['selected_models']}")
for item in sel_meta["details"]:
    print(f"  Rank {item['rank']}: {item['model']} (Inner Acc = {item['inner_accuracy_mean']:.4f} +/- {item['inner_accuracy_std']:.4f})")

### 4. Dedicated Top-3 Model Validation
Comprehensive evaluation of the selected top models across the 5 outer test folds. Reporting incorporates class balance-appropriate metrics: Balanced Accuracy, Macro F1, Sensitivity, Specificity, ROC-AUC, PR-AUC, MCC, and Brier Score Loss.

In [ ]:
top3_path = os.path.join("..", "results", "top3_validation", "top3_summary.csv")
top3_df = pd.read_csv(top3_path)
print("=== OUT-OF-FOLD TOP-3 VALIDATION METRICS ===")
top3_df[[
    "model", "accuracy_mean", "accuracy_std", "balanced_accuracy_mean", "balanced_accuracy_std",
    "recall_sensitivity_mean", "recall_sensitivity_std", "specificity_mean", "specificity_std",
    "f1_mean", "f1_std", "roc_auc_mean", "roc_auc_std"
]]

In [ ]:
# Display aggregated out-of-fold confusion matrices (long format written by step 17)
cm_path = os.path.join("..", "results", "top3_validation", "top3_confusion_matrices.csv")
cm_df = pd.read_csv(cm_path)
print("=== AGGREGATED OUT-OF-FOLD CONFUSION MATRICES (all 51 subjects, each tested once) ===")
for model, g in cm_df.groupby("model"):
    m = g.pivot_table(index="actual", columns="predicted", values="count", aggfunc="sum").fillna(0).astype(int)
    print(f"\nModel: {model}")
    print(m.to_string())


### 5. ROC and Precision-Recall Curves
Out-of-fold discrimination curves for the selected top models.

In [ ]:
fig_dir = os.path.join("..", "results", "figures")
roc_fig = os.path.join(fig_dir, "Figure_6_ROC_curves.png")
pr_fig = os.path.join(fig_dir, "Figure_7_Precision_Recall_curves.png")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
if os.path.exists(roc_fig):
    axes[0].imshow(Image.open(roc_fig))
    axes[0].axis("off")
    axes[0].set_title("ROC Curves")
if os.path.exists(pr_fig):
    axes[1].imshow(Image.open(pr_fig))
    axes[1].axis("off")
    axes[1].set_title("Precision-Recall Curves")
plt.tight_layout()
plt.show()

### 6. Representation Ablation Study
Tests whether repetition-aware descriptors (SD, CV, slope, early-late change, monotonicity across rep1-rep8) add discriminative information beyond static per-subject means. **Read the result as reported: the study does not demonstrate an advantage** (see the paired-fold Wilcoxon table below; with only 5 paired folds the test is under-powered).

In [ ]:
ablation_path = os.path.join("..", "results", "ablation", "ablation_summary.csv")
ablation_df = pd.read_csv(ablation_path)
print("=== FEATURE REPRESENTATION ABLATION RESULTS ===")
ablation_df[["representation", "accuracy_mean", "accuracy_std", "f1_mean", "f1_std", "roc_auc_mean", "roc_auc_std"]]
print(pd.read_csv(os.path.join('..','results','ablation','ablation_statistical_comparison.csv')).round(3).to_string(index=False))


### 7. Explainable AI: Out-of-Fold SHAP Analysis

In [ ]:
shap_fig = os.path.join(fig_dir, "Figure_9_SHAP_beeswarm.png")
if os.path.exists(shap_fig):
    plt.figure(figsize=(10, 8))
    plt.imshow(Image.open(shap_fig))
    plt.axis("off")
    plt.title("Out-of-Fold SHAP Beeswarm Summary")
    plt.show()

### 8. Leakage and Methodology Audit Verification
Automated checks confirming zero subject overlap, fit-on-train-only preprocessing, inner-CV-only model selection, no identifier predictors, plus the empirical label-permutation test (Section 10).

In [ ]:
audit_path = os.path.join("..", "results", "leakage_audit", "leakage_audit.csv")
audit_df = pd.read_csv(audit_path)
print("=== LEAKAGE & METHODOLOGY AUDIT ===")
for _, row in audit_df.iterrows():
    print(f"[{row['status']}] {row['check']}: {row['details']}")

### 9. Primary (selection-unbiased) estimate, repeated CV and uncertainty

Table 1b is the **fully nested** estimate: model family and hyperparameters are chosen inside each outer training fold. Table 2 (the 3 selected families) is optimistic for the procedure, because the ranking used all folds. Table 3 re-partitions the same 51 subjects, so it shows split sensitivity only. 'Final test' means outer CV folds; there is no independent test set. Recording-site confounding is unresolved.

In [ ]:
print(open(os.path.join('..','results','final_comparison','results_table.md'),encoding='utf-8').read())

### 10. Empirical leakage check (label permutation) and split-integrity verification
If any leakage existed, models trained on shuffled labels would score above chance. Subject-ID number alone predicting the label shows why identifiers are excluded from the predictors.

In [ ]:
print(pd.read_csv(os.path.join('..','results','leakage_audit','label_permutation_test.csv')).T.to_string(header=False))

### 11. Functional Resilience Index (rep1–rep8) and LIME

FRI, Consistency Score, Movement Stability Index and Adaptive Motor Control Index are computed from each child's own eight repetitions (label-free). LIME explanations are out-of-fold and compared with SHAP.

In [ ]:
res = os.path.join('..','results','functional_resilience')
print(pd.read_csv(os.path.join(res,'resilience_group_comparison.csv')).round(3).to_string(index=False))
print(pd.read_csv(os.path.join(res,'resilience_incremental_summary.csv')).round(3).to_string(index=False))

In [ ]:
lime_dir = os.path.join('..','results','lime_oof')
print(pd.read_csv(os.path.join(lime_dir,'lime_shap_agreement.csv')).T.to_string(header=False))
print(pd.read_csv(os.path.join(lime_dir,'oof_lime_summary.csv')).head(15)[['feature','mean_abs_lime','mean_abs_shap','lime_rank','shap_rank']].round(4).to_string(index=False))

### 12. PRIMARY analysis: coordinate-only features (84 predictors) and feature-set audit
The primary model uses **key-point coordinate-derived features + target-object position error only (84 predictors)**. 2D video-derived joint-angle features (168-predictor `angle_enhanced`) are a labelled secondary analysis; `coordinate_no_position_error` (78) is the target-object ablation. Counts and modality assignments are verified empirically in step 32 (angle/object columns perturbed across all 408 recordings).

In [ ]:
res=os.path.join('..','results')
print(pd.read_csv(os.path.join(res,'feature_audit','feature_set_sizes.csv')).to_string(index=False))
fa=pd.read_csv(os.path.join('..','data','processed','feature_audit.csv'))
print(fa.groupby('source_modality').size())
print(fa.head(10).to_string(index=False))

### 13. Headline: fully nested estimate, repeated whole-procedure runs, feature-set comparison
Headline (pre-specified seed-42 partition) is the fully nested estimate of the whole selection procedure. Repeated partitions re-run the complete procedure on the same 51 subjects (split sensitivity only). Nothing here is external validation.

In [ ]:
print(pd.read_csv(os.path.join(res,'nested_selection','nested_selection_ci.csv')).round(3).to_string(index=False))
print(pd.read_csv(os.path.join(res,'repeated_selection','repeated_selection_summary.csv')).round(3).to_string(index=False))
print(pd.read_csv(os.path.join(res,'feature_sets','feature_set_headline_comparison.csv')).round(3).T.to_string(header=False))
print(pd.read_csv(os.path.join(res,'feature_sets','paired_subject_level_comparison.csv')).round(3).to_string(index=False))

### 14. Acquisition / site confound (unresolved)
No site, session or camera variable exists in the data; subject-number blocks are structured by group. High accuracy may partly reflect recording setup. Independent, site-balanced validation is required.

In [ ]:
print(open(os.path.join(res,'confound_analysis','confound_summary.txt'),encoding='utf-8').read())

### 15. Per-fold top-3 selection and duration audit

In [ ]:
print(pd.read_csv(os.path.join(res,'nested_top3_per_fold','per_fold_top3_pooled_metrics.csv')).query('metric=="Accuracy"').round(3).to_string(index=False))
print(open(os.path.join(res,'duration_audit','duration_audit_summary.txt'),encoding='utf-8').read()[:900])

## Added analyses (steps 39-43)
Explanations of the nested winners, importance across feature sets, FRI components, curated figures. Exploratory-forest SHAP/LIME vs nested-pipeline permutation importance are different objects.

In [ ]:
import pandas as pd
R='../results/'
pd.read_csv(R+'nested_pipeline_explanations/nested_pipeline_importance.csv').head(10)

In [ ]:
pd.read_csv(R+'importance_across_feature_sets/rank_agreement_between_sets.csv')

In [ ]:
pd.read_csv(R+'functional_resilience/fri_component_analysis.csv').round(3)

In [ ]:
from IPython.display import Image, display
for f in ['F1_evidence_hierarchy','F3_feature_set_ablation','F5_FRI_components','F7_nested_pipeline_importance']:
    display(Image(R+'figures_final/'+f+'.png'))